<a href="https://colab.research.google.com/github/somendrew/LMS_Image_Classification/blob/main/LMS_Offical_Image_Stealth_Scrapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
from google.colab import auth
import gspread
from google.auth import default

# Authenticate the user
auth.authenticate_user()

# Get credentials and authorize gspread
creds, _ = default()
gc = gspread.authorize(creds)

In [3]:
# Open the sheet by its name
spreadsheet = gc.open('  P0.1 Sports {Exploratory analysis} || FIFA || Somendra ')

# Select the first worksheet (tab)
worksheet = spreadsheet.sheet1

In [4]:
worksheet

<Worksheet 'P0.1 Curation' id:0>

In [5]:
# Get all values from the worksheet
rows = worksheet.get_all_values()

# Convert to a DataFrame (using the first row as headers)
df = pd.DataFrame(rows[1:], columns=rows[0])

# View the first few rows
df.head()

,MID,MID,Athlete Name,Getty Image link 1,Image 1,Getty Image link 2,Image 2,Getty Image link 3,Image 3,Getty Image link 4,Image 4,Getty Image link 5,Image 5,Screenshot,Comment,Image Status,Assigned To,Accuracy,Status,Date of analysis
0,/g/11bxc63plb,/g/11bxc63plb,Ben Hall,,,,,,,,,,,,,,,,,
1,/g/11c2nynb6p,/g/11c2nynb6p,Abdeirahman Moustafa,,,,,,,,,,,,,,,,,
2,/g/11c3yw1b65,/g/11c3yw1b65,Marko Tolić,,,,,,,,,,,,,,,,,
3,/g/11f03ymw7t,/g/11f03ymw7t,Danijel Šturm,,,,,,,,,,,,,,,,,
4,/g/11f5v0_slc,/g/11f5v0_slc,Daniel Håkans,,,,,,,,,,,,,,,,,


In [8]:
#Taking sample df

new_df = df[:50]
new_df.shape

(50, 20)

In [7]:
new_df.head()

,MID,MID,Athlete Name,Getty Image link 1,Image 1,Getty Image link 2,Image 2,Getty Image link 3,Image 3,Getty Image link 4,Image 4,Getty Image link 5,Image 5,Screenshot,Comment,Image Status,Assigned To,Accuracy,Status,Date of analysis
0,/g/11bxc63plb,/g/11bxc63plb,Ben Hall,,,,,,,,,,,,,,,,,
1,/g/11c2nynb6p,/g/11c2nynb6p,Abdeirahman Moustafa,,,,,,,,,,,,,,,,,
2,/g/11c3yw1b65,/g/11c3yw1b65,Marko Tolić,,,,,,,,,,,,,,,,,
3,/g/11f03ymw7t,/g/11f03ymw7t,Danijel Šturm,,,,,,,,,,,,,,,,,
4,/g/11f5v0_slc,/g/11f5v0_slc,Daniel Håkans,,,,,,,,,,,,,,,,,


## Scraper

## Iterate through DataFrame and fill the links

In [9]:
from curl_cffi import requests
from bs4 import BeautifulSoup
import time
player_images = []

def stealth_scrape(entity):
    url = f"https://www.gettyimages.com/photos/{entity.replace(' ', '-')}"

    response = requests.get(url, impersonate="chrome120")

    images_found = []  # Local list instead of global

    if response.status_code == 200:
        soup = BeautifulSoup(response.content, "html.parser")
        images = soup.find_all('img')
        for img in images[:20]:
            src = img.get('src')
            if src:  # Also guard against None src values
                images_found.append(src)
    else:
        print(f"Blocked! Status Code: {response.status_code}")

    return images_found  # Always return a list (empty if blocked)


# Iterate through the DataFrame
print("Starting bulk scrape...")

for index, row in new_df.iterrows():
    entity = row['Athlete Name']
    query = f'{entity} poses for a Portrait'

    print(f"Processing: {entity}")

    # Fetch links
    player_images = stealth_scrape(query)

    # Splitting entity to check for individual keywords (case-insensitive) present in the link
    keywords = [k.lower() for k in entity.split()]

    #Defining number of getty columns we have in sheet
    max_columns = 5

    filtered_images = [ link for link in player_images if all(word in link.lower() for word in keywords)]

    print(f"Total images found: {len(filtered_images)}")

    # Images we need (5)
    images_to_save = filtered_images[:max_columns]

    # Fill the dataframe columns
    for i, link in enumerate(images_to_save):
        new_df.at[index, f'Getty Image link {i+1}'] = link

    # Respectful delay to avoid IP bans
    time.sleep(2)



Starting bulk scrape...
Processing: Ben Hall
Total images found: 1
Processing: Abdeirahman Moustafa
Total images found: 0
Processing: Marko Tolić
Total images found: 0
Processing: Danijel Šturm
Total images found: 0
Processing: Daniel Håkans
Total images found: 0
Processing: Ian Hoffmann
Total images found: 0
Processing: Fabian Wohlmuth
Total images found: 0
Processing: Mamady Diarra
Total images found: 0
Processing: Alexandre Llovet
Total images found: 0
Processing: Camutanga
Total images found: 0
Processing: Fernandinho
Total images found: 15
Processing: Albert Dabiqaj
Total images found: 0
Processing: João Paulo Fernandes
Total images found: 0
Processing: Ilir Mustafa
Total images found: 0
Processing: Yannick Semedo
Total images found: 0
Processing: Hanus Sørensen
Total images found: 0
Processing: Milot Avdyli
Total images found: 0
Processing: Ángel de la Torre
Total images found: 0
Processing: Alaa Ghram
Total images found: 0
Processing: David Rodríguez
Total images found: 0
Proces

In [ ]:
new_df

In [11]:
from google.colab import files

#  Save the file first
new_df.to_csv('test_set2.csv', index=False)

# Trigger the browser download
files.download('test_set2.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>